# StepManAI — Generate a stepchart

AI stepchart generator trained on ~1800 official DDR simfiles.

1. Run the **Setup** cell (needs trained checkpoints in `Drive/StepManAI/checkpoints/`).
2. Fill in the **form**: paste a Spotify link (title/artist auto-fill) *or* leave it empty to upload an audio file. Pick a level **1–19** (DDR scale).
3. Run the cell — you get a `.zip`; unzip it into your `Songs\StepManAI\` folder and reload songs.


In [ ]:
#@title Setup
!pip install -q soundfile spotdl
import os
if not os.path.exists('/content/StepManAI'):
    !git clone -q https://github.com/Mrman67/StepManAI.git /content/StepManAI
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/StepManAI/checkpoints'
assert os.path.exists(CKPT + '/placement.pt'), 'No checkpoints found — run the training notebook first.'
os.environ['STEPMANAI_CKPT'] = CKPT
print('ready')

In [ ]:
#@title Generate
spotify_link = "" #@param {type:"string"}
title = "" #@param {type:"string"}
artist = "" #@param {type:"string"}
level = 12 #@param {type:"slider", min:1, max:19, step:1}
all_difficulties = False #@param {type:"boolean"}
temperature = 0.9 #@param {type:"slider", min:0.5, max:1.3, step:0.05}
seed = 0 #@param {type:"integer"}

import glob, os, shutil, subprocess, sys
if spotify_link.strip():
    audio = spotify_link.strip()
else:
    from google.colab import files
    print('Spotify link empty — upload an audio file (.mp3/.ogg/.wav):')
    up = files.upload()
    assert up, 'no file uploaded'
    audio = '/content/' + list(up)[0]
    shutil.move(list(up)[0], audio) if not os.path.exists(audio) else None

shutil.rmtree('/content/output', ignore_errors=True)
lv = 'all' if all_difficulties else str(level)
cmd = [sys.executable, '/content/StepManAI/generate.py', audio, '-l', lv,
       '--out', '/content/output', '--temperature', str(temperature)]
if title.strip(): cmd += ['-t', title.strip()]
if artist.strip(): cmd += ['-a', artist.strip()]
if seed: cmd += ['--seed', str(seed)]
r = subprocess.run(cmd)
assert r.returncode == 0, 'generation failed (see output above)'

song = sorted(glob.glob('/content/output/*'), key=os.path.getmtime)[-1]
zpath = shutil.make_archive('/content/' + os.path.basename(song), 'zip',
                            root_dir='/content/output', base_dir=os.path.basename(song))
from google.colab import files
files.download(zpath)
print('\nUnzip into OutFox Songs\\StepManAI\\ and reload songs.')